# B6.0 — Setup, data checks, baseline subsets and the pre-declared B6 rule file

Runs on a **CPU runtime** (no GPU needed). No method is run and no prediction is made here.

| Part | What |
|---|---|
| A | Paths, repo, loaders |
| B | Data checks — VOC 2007 test (checksum, extraction, both "difficult" conventions), CARPK (verified Kaggle copy), FSC-147 test, OmniCount-191 test copy |
| C | **Draft** instance-vs-area class groups for OmniCount-191 (from class **names** only) → you review and approve |
| D | Fixed baseline subsets (seed 0): VOC 500-image standard-protocol subset; OmniCount Fruits test subset |
| E | **Rule file** — written once, only after the class groups are approved (`CLASS_GROUPS_APPROVED = True`); refuses to change afterwards |

Archives stay on Drive (`datasets/`); each session extracts them to the local disk (`/content/data/`), which is fast to read.

In [1]:
# ============================================================
# Step A.1 — Mount Drive, paths, repo
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, re, shutil, subprocess, sys, tarfile, zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
DATASETS = PROJECT_ROOT / "datasets"
RESULTS_ROOT = PROJECT_ROOT / "results"
OUT_DIR = RESULTS_ROOT / "B6_0_Setup_and_Rules"; OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/data"); LOCAL.mkdir(parents=True, exist_ok=True)
VOC_TAR = DATASETS / "VOCtest_06-Nov-2007.tar"
VOC_MD5 = "b6e924de25625d8de591ea690078ad9f"                 # torchvision's published checksum
CARPK_ZIP = DATASETS / "CARPK_kaggle_coco.zip"
FSC147_RAW = DATASETS / "FSC147" / "raw"
OMNICOUNT_ROOT = DATASETS / "OmniCount-191"

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
subprocess.run(["rm", "-rf", str(REPO_DIR)])
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR))
import src.datasets as D
for name in ["CARPKDataset", "VOC2007CountingDataset", "FSC147Dataset", "OmniCount191Dataset"]:
    assert hasattr(D, name), f"src.datasets.{name} missing — is the repo up to date?"
print("Repo:", REPO_COMMIT[:10], "| archives on Drive:",
      {p.name: f"{p.stat().st_size / 1e6:.0f} MB" for p in [VOC_TAR, CARPK_ZIP] if p.exists()})
assert VOC_TAR.exists() and CARPK_ZIP.exists(), "Archive(s) missing in datasets/"
INV = {}

Mounted at /content/drive
Repo: 60a6fb2f9a | archives on Drive: {'VOCtest_06-Nov-2007.tar': '451 MB', 'CARPK_kaggle_coco.zip': '2082 MB'}


In [2]:
# ============================================================
# Step B.1 — PASCAL VOC 2007 test: checksum, extraction, statistics (both "difficult" conventions)
# ============================================================
h = hashlib.md5()
with open(VOC_TAR, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 22), b""):
        h.update(chunk)
assert h.hexdigest() == VOC_MD5, f"VOC archive checksum {h.hexdigest()} != {VOC_MD5}"
print("VOC archive checksum OK:", VOC_MD5)
VOC_LOCAL = LOCAL / "voc2007"
if not (VOC_LOCAL / "VOCdevkit" / "VOC2007" / "ImageSets").exists():
    with tarfile.open(VOC_TAR) as t:
        t.extractall(VOC_LOCAL)
voc_all = D.VOC2007CountingDataset(VOC_LOCAL, "test", include_difficult=True, load_images=False)
rows = []
for i in range(len(voc_all)):
    s = voc_all[i]
    for c, v in s["categories"].items():
        rows.append({"image_id": s["image_id"], "class": c, "count_all": v["count_all"], "count_nondifficult": v["count_nondifficult"]})
vq = pd.DataFrame(rows)
n_jpg = len(list((voc_all.voc_dir / "JPEGImages").glob("*.jpg")))
INV["voc"] = {"images": len(voc_all), "jpeg_files": n_jpg, "objects_all": int(vq["count_all"].sum()),
              "objects_nondifficult": int(vq["count_nondifficult"].sum()),
              "present_queries_all": int((vq["count_all"] > 0).sum()),
              "present_queries_nondifficult": int((vq["count_nondifficult"] > 0).sum()),
              "standard_queries": len(voc_all) * len(D.VOC_CLASSES),
              "max_count_per_class_image": int(vq["count_all"].max())}
assert len(voc_all) == 4952 and n_jpg == 4952, INV["voc"]
print(INV["voc"])
print("\nPer class (all / non-difficult):")
display(vq.groupby("class")[["count_all", "count_nondifficult"]].sum().T)
vq.to_csv(OUT_DIR / "B6_0_voc_test_gt_counts.csv", index=False)

VOC archive checksum OK: b6e924de25625d8de591ea690078ad9f


/tmp/ipykernel_1412/3335764157.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(VOC_LOCAL)


{'images': 4952, 'jpeg_files': 4952, 'objects_all': 14976, 'objects_nondifficult': 12032, 'present_queries_all': 7632, 'present_queries_nondifficult': 7013, 'standard_queries': 99040, 'max_count_per_class_image': 41}

Per class (all / non-difficult):


class,aeroplane,bicycle,bird,boat,bottle,bus,car,cat,chair,cow,diningtable,dog,horse,motorbike,person,pottedplant,sheep,sofa,train,tvmonitor
count_all,311,389,576,393,657,254,1541,370,1374,329,299,530,395,369,5227,592,311,396,302,361
count_nondifficult,285,337,459,263,469,213,1201,358,756,244,206,489,348,325,4528,480,242,239,282,308


In [3]:
# ============================================================
# Step B.2 — CARPK (verified Kaggle COCO copy): extraction and statistics
# ============================================================
CARPK_LOCAL = LOCAL / "carpk"
if not any(CARPK_LOCAL.rglob("test_annotations.json")):
    with zipfile.ZipFile(CARPK_ZIP) as z:
        z.extractall(CARPK_LOCAL)
EXPECTED = {"test": (459, 47500, 2, 188), "train": (989, 42274, 1, 87)}    # verified 26 Oct (Kaggle = Deep Lake)
for split, exp in EXPECTED.items():
    ds = D.CARPKDataset(CARPK_LOCAL, split, load_images=False)
    c = [ds[i]["categories"]["car"]["count"] for i in range(len(ds))]
    files = sum(1 for _ in ds.image_dir.glob("*.png"))
    got = (len(ds), int(sum(c)), int(min(c)), int(max(c)))
    INV[f"carpk_{split}"] = {"images": got[0], "png_files": files, "cars": got[1], "min": got[2], "max": got[3]}
    print(split, INV[f"carpk_{split}"], "OK" if got == exp and files == exp[0] else f"MISMATCH (expected {exp})")
    assert got == exp and files == exp[0]
ds_c = D.CARPKDataset(CARPK_LOCAL, "test", load_images=False)
pd.DataFrame([{"image_id": ds_c[i]["image_id"], "gt_count": ds_c[i]["categories"]["car"]["count"]} for i in range(len(ds_c))]
             ).to_csv(OUT_DIR / "B6_0_carpk_test_gt_counts.csv", index=False)

test {'images': 459, 'png_files': 459, 'cars': 47500, 'min': 2, 'max': 188} OK
train {'images': 989, 'png_files': 989, 'cars': 42274, 'min': 1, 'max': 87} OK


In [4]:
# ============================================================
# Step B.3 — FSC-147 test (1,190 images) and the S1–S6 boundaries (from FSC-147 val, B4)
# ============================================================
fsc = D.FSC147Dataset(FSC147_RAW, split="test", load_images=False)
ft = pd.DataFrame([{"dataset_index": i, "image_id": fsc[i]["image_id"], "category": next(iter(fsc[i]["categories"])),
                    "gt_count": next(iter(fsc[i]["categories"].values()))["count"]} for i in range(len(fsc))])
val = pd.read_csv(RESULTS_ROOT / "B4_HeldOut_FSC147_Validation" / "partC_evaluation" / "B4_C_fullval_per_image.csv")
bounds = val.groupby("density_stratum")["gt_count"].agg(["min", "max"]).sort_index()
print("S1–S6 boundaries from FSC-147 val (B4):"); display(bounds)
uppers = list(bounds["max"])[:-1]
def stratum(n):
    for lab, up in zip(bounds.index, uppers):
        if n <= up:
            return lab
    return bounds.index[-1]
ft["density_stratum"] = ft["gt_count"].apply(stratum)
INV["fsc147_test"] = {"images": len(ft), "classes": ft["category"].nunique(), "gt_min": int(ft["gt_count"].min()),
                      "gt_max": int(ft["gt_count"].max()), "per_stratum": ft["density_stratum"].value_counts().sort_index().to_dict()}
assert len(ft) == 1190, len(ft)
print(INV["fsc147_test"])
ft.to_csv(OUT_DIR / "B6_0_fsc147_test_gt_counts.csv", index=False)
bounds.to_csv(OUT_DIR / "B6_0_fsc147_strata_boundaries.csv")

S1–S6 boundaries from FSC-147 val (B4):


,min,max
density_stratum,,
S1_0_20,7,12
S2_20_40,12,20
S3_40_60,20,37
S4_60_80,37,76
S5_80_95,77,215
S6_95_100,216,2092


{'images': 1190, 'classes': 29, 'gt_min': 8, 'gt_max': 3701, 'per_stratum': {'S1_0_20': 142, 'S2_20_40': 186, 'S3_40_60': 277, 'S4_60_80': 291, 'S5_80_95': 247, 'S6_95_100': 47}}


In [5]:
# ============================================================
# Step B.4 — OmniCount-191 test copy: domains, vocabulary, queries
# ============================================================
om = D.OmniCount191Dataset(OMNICOUNT_ROOT, split="test", load_images=False)
VOCAB_TEST = om.domain_vocabulary
recs = []
for i in range(len(om)):
    s = om[i]
    recs.append({"idx": i, "image_id": s["image_id"], "domain": s["domain"], "image_path": s["image_path"],
                 "gt": json.dumps({c: v["count"] for c, v in s["categories"].items()})})
ot = pd.DataFrame(recs)
inv_om = ot.groupby("domain").agg(images=("idx", "size"))
inv_om["vocabulary"] = [len(VOCAB_TEST[d]) for d in inv_om.index]
inv_om["present_queries"] = ot.groupby("domain")["gt"].apply(lambda g: sum(len(json.loads(x)) for x in g))
inv_om["standard_queries"] = inv_om["images"] * inv_om["vocabulary"]
display(inv_om)
assert "Household" not in inv_om.index, "Household should not be in the test split"
INV["omnicount_test"] = {"images": len(ot), "domains": list(inv_om.index), "classes": int(inv_om["vocabulary"].sum()),
                         "present_queries": int(inv_om["present_queries"].sum()), "standard_queries": int(inv_om["standard_queries"].sum())}
print(INV["omnicount_test"])
ot.to_csv(OUT_DIR / "B6_0_omnicount_test_images.csv", index=False)

,images,vocabulary,present_queries,standard_queries
domain,,,,
Birds,10,3,29,30
Fruits,303,8,1226,2424
Pets,11,18,53,198
Satellite,127,23,130,2921
Supermarket,251,52,293,13052
Urban,1000,12,1756,12000
Wild,255,4,267,1020


{'images': 1957, 'domains': ['Birds', 'Fruits', 'Pets', 'Satellite', 'Supermarket', 'Urban', 'Wild'], 'classes': 120, 'present_queries': 3754, 'standard_queries': 31645}


In [6]:
# ============================================================
# Step C.1 — DRAFT instance-vs-area class groups (from class NAMES only — no counts, no predictions)
#   area = annotated as a region / stuff-like extent (parking area, farmland, field, school, ...)
#   instance = countable objects (car, apple, person, ...)
# ============================================================
AREA_WORDS = ["area", "land", "field", "ground", "court", "lot", "park", "playground", "farm", "desert", "forest",
              "meadow", "lake", "river", "pond", "sea", "beach", "road", "highway", "roundabout", "intersection",
              "stadium", "school", "airport", "harbor", "harbour", "port", "station", "track", "golf", "baseball",
              "basketball", "tennis", "soccer", "football", "residential", "industrial", "commercial", "houses",
              "building", "bareland", "grass", "vegetation", "water", "dam", "bridge"]
def suggest(cls):
    words = re.findall(r"[a-z]+", cls.lower())
    hit = [w for w in AREA_WORDS if any(x == w or x.startswith(w) for x in words)]
    return ("area" if hit else "instance"), ",".join(hit)
rows = []
for d in sorted(VOCAB_TEST):
    for c in VOCAB_TEST[d]:
        g, why = suggest(c)
        rows.append({"domain": d, "class": c, "prompt": D.omnicount_prompt_name(c), "group": g,
                     "keyword": why, "review": "CHECK" if (d == "Satellite" or g == "area") else ""})
groups = pd.DataFrame(rows)
DRAFT = OUT_DIR / "B6_0_omnicount_class_groups_DRAFT.csv"
groups.to_csv(DRAFT, index=False)
print(f"Draft written: {DRAFT.name} | {len(groups)} classes | area: {int((groups['group'] == 'area').sum())}")
pd.set_option("display.max_rows", 300)
display(groups[groups["review"] == "CHECK"])
print("\nNEXT: review the draft (all Satellite classes and every 'area' suggestion), correct the 'group' column,")
print("save it as B6_0_omnicount_class_groups_FINAL.csv in the same folder, then set CLASS_GROUPS_APPROVED = True in Step E.1.")

Draft written: B6_0_omnicount_class_groups_DRAFT.csv | 120 classes | area: 18


,domain,class,prompt,group,keyword,review
29,Satellite,Baseball field,baseball field,area,"field,baseball",CHECK
30,Satellite,Church,church,instance,,CHECK
31,Satellite,Farmlands,farmlands,area,farm,CHECK
32,Satellite,Forest,forest,area,forest,CHECK
33,Satellite,Mountains,mountains,instance,,CHECK
34,Satellite,Pond,pond,area,pond,CHECK
35,Satellite,Railway Station,railway station,area,station,CHECK
36,Satellite,River,river,area,river,CHECK
37,Satellite,School,school,area,school,CHECK
38,Satellite,aeroplanes,aeroplanes,instance,,CHECK



NEXT: review the draft (all Satellite classes and every 'area' suggestion), correct the 'group' column,
save it as B6_0_omnicount_class_groups_FINAL.csv in the same folder, then set CLASS_GROUPS_APPROVED = True in Step E.1.


In [7]:
# ============================================================
# Step D.1 — Fixed baseline subsets (seed 0)
# ============================================================
SEED = 0
voc_ids = sorted(voc_all.image_ids)
voc_sub = sorted(pd.Series(voc_ids).sample(n=500, random_state=SEED).tolist())
fruits = ot[ot["domain"] == "Fruits"]["image_id"].tolist()
SUBSETS = {"voc_standard_500": voc_sub, "omnicount_fruits_test": fruits}
with open(OUT_DIR / "B6_0_baseline_subsets.json", "w") as f:
    json.dump({"seed": SEED, **SUBSETS}, f, indent=1)
print({k: len(v) for k, v in SUBSETS.items()}, "| VOC subset sample:", voc_sub[:5])

{'voc_standard_500': 500, 'omnicount_fruits_test': 303} | VOC subset sample: ['000022', '000058', '000062', '000076', '000082']


In [8]:
# ============================================================
# Step D.2 — Save the inventory
# ============================================================
with open(OUT_DIR / "B6_0_inventory.json", "w") as f:
    json.dump({"date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "repo_commit": REPO_COMMIT, **INV}, f, indent=2)
print("Saved to:", OUT_DIR); print(json.dumps(INV, indent=1)[:2000])

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B6_0_Setup_and_Rules
{
 "voc": {
  "images": 4952,
  "jpeg_files": 4952,
  "objects_all": 14976,
  "objects_nondifficult": 12032,
  "present_queries_all": 7632,
  "present_queries_nondifficult": 7013,
  "standard_queries": 99040,
  "max_count_per_class_image": 41
 },
 "carpk_test": {
  "images": 459,
  "png_files": 459,
  "cars": 47500,
  "min": 2,
  "max": 188
 },
 "carpk_train": {
  "images": 989,
  "png_files": 989,
  "cars": 42274,
  "min": 1,
  "max": 87
 },
 "fsc147_test": {
  "images": 1190,
  "classes": 29,
  "gt_min": 8,
  "gt_max": 3701,
  "per_stratum": {
   "S1_0_20": 142,
   "S2_20_40": 186,
   "S3_40_60": 277,
   "S4_60_80": 291,
   "S5_80_95": 247,
   "S6_95_100": 47
  }
 },
 "omnicount_test": {
  "images": 1957,
  "domains": [
   "Birds",
   "Fruits",
   "Pets",
   "Satellite",
   "Supermarket",
   "Urban",
   "Wild"
  ],
  "classes": 120,
  "present_queries": 3754,
  "standard_queries": 31645
 }
}


In [10]:
corr = {("Satellite", "Mountains"): "area", ("Satellite", "bridge"): "instance",
        ("Satellite", "building"): "instance", ("Satellite", "triangular building"): "instance",
        ("Supermarket", "Watermelon"): "instance"}
fin = groups.copy()
for (d, c), g in corr.items():
    sel = (fin["domain"] == d) & (fin["class"] == c)
    assert sel.sum() == 1, (d, c)
    fin.loc[sel, "group"] = g
fin.to_csv(OUT_DIR / "B6_0_omnicount_class_groups_FINAL.csv", index=False)
print(fin["group"].value_counts().to_dict())
print(sorted(fin.loc[fin["group"] == "area", "class"]))

{'instance': 105, 'area': 15}
['Baseball field', 'Farmlands', 'Forest', 'Mountains', 'Pond', 'Railway Station', 'River', 'School', 'bareland', 'beach', 'desert', 'houses', 'parking area', 'playground', 'road']


In [12]:
# ============================================================
# Step E.1 — Pre-declared B6 rule file (written ONCE, after the class groups are approved)
# ============================================================
CLASS_GROUPS_APPROVED = True          # set True only after saving B6_0_omnicount_class_groups_FINAL.csv
RULE_PATH = OUT_DIR / "B6_rule_file.json"
FINAL = OUT_DIR / "B6_0_omnicount_class_groups_FINAL.csv"
if not CLASS_GROUPS_APPROVED:
    raise SystemExit("Class groups not approved yet — review the DRAFT, save the FINAL csv, then set CLASS_GROUPS_APPROVED = True.")
fin = pd.read_csv(FINAL)
assert set(zip(fin["domain"], fin["class"])) == set(zip(groups["domain"], groups["class"])), "FINAL must list exactly the test classes"
assert set(fin["group"]) <= {"instance", "area"}, "group must be 'instance' or 'area'"
RULE = {
    "phase": "B6 test evaluation",
    "frozen_module": {"candidates_v2_commit": "2086087c5e", "pipelines": {"primary": "GPV_R4+P[MixA] (counting_gpv_r4_p_mixa)",
                                                                           "secondary": "GPV_K-n+P[LM0] (counting_gpv_kn_p_lm0)"}},
    "datasets": {
        "FSC-147": {"split": "test", "images": INV["fsc147_test"]["images"], "mode": "single (v1.1; v1 derived)"},
        "CARPK": {"split": "test", "images": INV["carpk_test"]["images"], "source": "Kaggle COCO conversion verified vs Deep Lake "
                  "(47,500 test cars); official files compared per image if they arrive", "gt": "number of boxes, as given",
                  "mode": "single (v1.1; v1 derived)"},
        "OmniCount-191": {"split": "test (our early-access copy; Roboflow split)", "images": INV["omnicount_test"]["images"],
                          "vocabulary": "test split's own domain vocabulary", "mode": "multi (U_both)"},
        "PASCAL VOC 2007": {"split": "test (Count-test, 4,952 images)", "mode": "multi (U_both)", "vocabulary": "all 20 classes",
                            "gt_primary": "all annotated objects (count_all)", "gt_secondary": "non-difficult objects only (sensitivity)",
                            "archive_md5": VOC_MD5}},
    "methods": {"ours": ["GPV_R4+P[MixA]", "GPV_K-n+P[LM0]"], "internal_reference": ["B0 (FSC-147, CARPK)"],
                "literature_anchor": ["vanilla SAM ViT-B everything mode (FSC-147, CARPK)"],
                "baselines": {"TFOC": "official code, SAM ViT-B, text template 'the photo of many <class>'; reproduction check FSC-147 text MAE 24.79",
                              "OmniCount": "official code, time-boxed; reproduction check FSC-147 MAE 21.46"},
                "baseline_failure": "report our re-run and the published figure, gap explained"},
    "baseline_scope": {"single_class": "full test sets",
                       "multi_label": {"present_classes_protocol": "full test sets",
                                       "standard_protocol": {"voc": "voc_standard_500 (seed 0)", "omnicount": "Fruits test subset"}},
                       "ours_also_on_subsets": True, "subset_file": "B6_0_baseline_subsets.json"},
    "metrics": {"single_class": {"primary": "MAE", "also": ["RMSE", "NAE", "SRE"]},
                "multi_label": {"primary": "standard mRMSE", "also": ["mRMSE-nz", "MAE_nz", "OmniCount-code metric (not comparable with the paper)"]}},
    "versions": "v1.1 (with ladder) headline for both systems; v1 derived from the same run (ladder-rescued images counted 0)",
    "statistics": {"paired_bootstrap": {"n": 10000, "seed": 0, "unit": "image"},
                   "comparisons": ["primary vs each baseline", "primary vs secondary"],
                   "on": {"single_class": "MAE", "multi_label": "MAE_nz"}},
    "confirmatory_breakdowns": {"fsc147": "S1–S6 by FSC-147 val boundaries (B6_0_fsc147_strata_boundaries.csv)",
                                "omnicount": ["per domain", "Fruits subset", "without Wild",
                                              "instance vs area classes (B6_0_omnicount_class_groups_FINAL.csv)"]},
    "density_analysis": {"unit": "one ground-truth count (image for single-class; image-class pair for multi-label, present classes)",
                         "bins": [[1, 1], [2, 5], [6, 10], [11, 20], [21, 50], [51, 100], [101, 200], [201, None]],
                         "metrics": ["counting accuracy = mean(max(0, 1 - |pred - gt| / gt)) x 100", "MAE", "NAE", "samples per bin"],
                         "all methods, all datasets": True},
    "prediction_format": "one JSON per image: {key, image_id, dataset, classes, counts: {method: {class: int}}, meta, diag}",
    "environment": "L4 throughout; pinned installs; detectron2 --no-build-isolation; environment recorded per run",
    "exploratory": "any analysis not listed here is reported as exploratory",
    "inventory": INV, "repo_commit": REPO_COMMIT,
}
if RULE_PATH.exists():
    saved = json.load(open(RULE_PATH))
    assert {k: v for k, v in saved.items() if k not in ("recorded_utc", "repo_commit")} == \
           {k: v for k, v in RULE.items() if k != "repo_commit"}, "Rule file already recorded and differs — stop."
    print("Rule file already recorded at", saved["recorded_utc"], "— unchanged")
else:
    with open(RULE_PATH, "w") as f:
        json.dump({**RULE, "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}, f, indent=2)
    print("Rule file recorded:", RULE_PATH)
print("Class groups:", fin["group"].value_counts().to_dict())

Rule file recorded: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B6_0_Setup_and_Rules/B6_rule_file.json
Class groups: {'instance': 105, 'area': 15}
